In [ ]:
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- 1. 定義列表 ---
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1"
]

proxy_list = [
    "103.111.144.150:80", 
    "192.168.1.1:8888", 
]

# --- 2. 隨機選取 ---
random_user_agent = random.choice(user_agents)
random_proxy = random.choice(proxy_list)

print(f"本次使用的 User-Agent: {random_user_agent}")
print(f"本次使用的代理 IP: {random_proxy}")

# --- 3. 設定選項 ---
edge_options = Options()

# 保持你原有的選項
# edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--no-sandbox")
edge_options.add_argument("--disable-dev-shm-usage")
# edge_options.add_argument("--headless=chrome")
edge_options.add_argument("--window-size=1920,1080")
edge_options.add_argument("--start-maximized")

# 加入隨機 User-Agent
edge_options.add_argument(f"user-agent={random_user_agent}")

# 加入隨機代理 IP
# edge_options.add_argument(f'--proxy-server={random_proxy}')

# --- 4. 啟動驅動 ---
driver = webdriver.Chrome(options=edge_options)

url = "http://www.baseball-reference.com/leagues/daily.cgi?user_team=&bust_cache=&type=p&lastndays=7&dates=fromandto&fromandto=2024-03-01.2024-11-30&level=mlb&franch=&stat=&stat_value=0"

driver.get(url)

wait = WebDriverWait(driver, 10)
button = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//span[normalize-space()='Share & Export']")
    )
)

# 4️⃣ 點擊
button.click()

wait = WebDriverWait(driver, 10)
button = driver.find_element(By.CSS_SELECTOR, "button[tip='Convert the table below to comma-separated values<br>suitable for use with Excel']")
driver.execute_script("arguments[0].click();", button)

df = driver.find_element(By.CSS_SELECTOR, "pre[id='csv_daily']")
s = df.text

driver.close()

In [ ]:
from io import StringIO
import pandas as pd

lst_s = s.split("\n")[4:]

print(lst_s)
print(lst_s[0].split(","))


In [ ]:
import pandas as pd
from io import StringIO

csv_data = "\n".join(lst_s)
df = pd.read_csv(StringIO(csv_data))

clear_df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print(clear_df)

clear_df.to_excel("pitching.xlsx", index=False)

In [ ]:
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- 1. 定義列表 ---
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1"
]

proxy_list = [
    "103.111.144.150:80", 
    "192.168.1.1:8888", 
]

# --- 2. 隨機選取 ---
random_user_agent = random.choice(user_agents)
random_proxy = random.choice(proxy_list)

print(f"本次使用的 User-Agent: {random_user_agent}")
print(f"本次使用的代理 IP: {random_proxy}")

# --- 3. 設定選項 ---
edge_options = Options()

# 保持你原有的選項
# edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--no-sandbox")
edge_options.add_argument("--disable-dev-shm-usage")
# edge_options.add_argument("--headless=chrome")
edge_options.add_argument("--window-size=1920,1080")
edge_options.add_argument("--start-maximized")

# 加入隨機 User-Agent
edge_options.add_argument(f"user-agent={random_user_agent}")

# 加入隨機代理 IP
# edge_options.add_argument(f'--proxy-server={random_proxy}')

# --- 4. 啟動驅動 ---
driver = webdriver.Chrome(options=edge_options)

url = "https://www.baseball-reference.com/players/gl.fcgi?id=yamamyo01&t=p&year=2024"

driver.get(url)

# df = driver.find_element(By.CSS_SELECTOR, "table[id='daily']")
# print(df.text)
wait = WebDriverWait(driver, 10)
button = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//span[normalize-space()='Share & Export']")
    )
)

# 4️⃣ 點擊
button.click()

button = driver.find_element(By.CSS_SELECTOR, "button[tip='Export table as <br>suitable for use with Excel']")
button.click()

driver.close()

In [ ]:
# ============================================
# MLB 投手特徵資料自動生成腳本（使用新版 pybaseball）
# ============================================
import pandas as pd
from datetime import timedelta
from pybaseball import playerid_lookup, statcast_pitcher, team_batting, pitching_stats_range

# ======== ⚙️ 參數設定 ========
pitcher_name = "Gerrit Cole"
pitcher_first = "Gerrit"
pitcher_last = "Cole"
year = 2024
start_date = f"{year}-01-01"
end_date = f"{year}-12-31"

# ======== 🔍 取得投手 MLBAM ID ========
pid_df = playerid_lookup(pitcher_last, pitcher_first)
mlbam_id = pid_df["key_mlbam"].values[0]
print(f"✅ {pitcher_name} MLBAM ID: {mlbam_id}")

# ======== ⚾ 取得逐球資料（近似逐場） ========
pitch_df = statcast_pitcher(start_date, end_date, mlbam_id)
# 你可能需要由逐球 → 逐場匯總
pitch_df["game_date"] = pd.to_datetime(pitch_df["game_date"])

# 依場次匯總基本數字（示範：IP_num、ER、Pit 球數）
agg = pitch_df.groupby("game_date").agg({
    "pitcher": "first",
    "IP": "count",         # 注意：需要敲定如何從逐球資料轉 IP
    "balls": "count",      # 只是示範
    "events": lambda x: (x=="home_run").sum()
}).reset_index().rename(columns={"IP": "Pit_count", "balls": "Total_balls", "events": "HR_count"})

# ======== 🥇 取得賽季級別資料 ========
season_df = pitching_stats_range(start_date, end_date)
# 篩選該投手
season_df = season_df[season_df["Name"] == pitcher_name]
season_era = season_df["ERA"].iloc[0]
season_whip = season_df["WHIP"].iloc[0]

# ======== 🥊 對手強度（球隊 OPS / wOBA） ========
bat_df = team_batting(year)
bat_df = bat_df.rename(columns={"Team": "Opp"})
# 待你合併 game_date → 對手變數

# ======== 📊 整理輸出 ========
feature_df = agg.copy()
feature_df["season_era"] = season_era
feature_df["season_whip"] = season_whip
feature_df["hand"] = "R"  # 手別視你補入

feature_df.to_excel(f"{pitcher_name.replace(' ', '_')}_features_{year}.xlsx", index=False)
print(f"✅ 已輸出 {pitcher_name} {year} 特徵資料 -> Excel 完成！")


In [ ]:
# ============================================
# MLB 投手特徵資料自動生成（pybaseball 2.x 版本）
# 依賽季逐球資料 → 匯總為逐場 → 計算特徵 → 輸出 Excel
# ============================================
import pandas as pd
import numpy as np
from datetime import timedelta
from pybaseball import playerid_lookup, statcast_pitcher, pitching_stats, team_batting

# ========= 使用者參數 =========
PITCHER_FIRST = "Yoshinobu"
PITCHER_LAST  = "Yamamoto"
YEAR = 2025
START_DT = f"{YEAR}-01-01"
END_DT   = f"{YEAR}-12-31"
OUTFILE  = f"{PITCHER_FIRST}_{PITCHER_LAST}_features_{YEAR}.xlsx".replace(" ", "_")

# ========= 取得 MLBAM 投手 ID =========
pid_df = playerid_lookup(PITCHER_LAST, PITCHER_FIRST)
if pid_df.empty:
    raise RuntimeError("找不到球員 ID，請確認姓名拼字。")
PITCHER_ID = int(pid_df["key_mlbam"].iloc[0])
print(f"MLBAM ID of {PITCHER_FIRST} {PITCHER_LAST}: {PITCHER_ID}")

# ========= 抓逐球資料（Statcast） =========
pitches = statcast_pitcher(START_DT, END_DT, PITCHER_ID)
if pitches.empty:
    raise RuntimeError("此年度沒有 Statcast 投球資料。")
pitches["game_date"] = pd.to_datetime(pitches["game_date"])

# ------- 由逐球 → 逐場匯總 -------
# 1) 每場投球數（pitches）= 該場列數
game_pcount = pitches.groupby("game_date").size().rename("pitches")

# 2) 出局數 → IP：利用 outs_when_up 的遞增量（僅統計投手在場時造成的出局）
def outs_from_half_inning(df):
    # 依正確時序排序
    df = df.sort_values(by=[
        'game_date', 'inning', 'inning_topbot', 'at_bat_number', 'pitch_number'
    ])

    # 基礎：同「局+半局」內的正向遞增量（0->1、1->2）
    base_delta = (
        df.groupby(['inning','inning_topbot'])['outs_when_up']
          .diff().clip(lower=0).fillna(0).astype(int)
    )

    # 補強：半局最後一球造成第3個出局（資料呈現為上一列=2，下一列換半局/換局且=0）
    prev_outs = df['outs_when_up'].shift(1)
    half_changed = (
        df['inning'].ne(df['inning'].shift(1)) |
        df['inning_topbot'].ne(df['inning_topbot'].shift(1))
    )
    extra_third = ((prev_outs == 2) & half_changed).astype(int)

    # 總出局數 = 同半局內遞增量 + 邊界補1
    outs_total = int(base_delta.sum() + extra_third.sum())
    return outs_total

# 其餘維持你的寫法
pitches = pitches.sort_values(by=[
    'game_date', 'inning', 'inning_topbot', 'at_bat_number', 'pitch_number'
], ascending=True)

outs_by_game = pitches.groupby("game_date").apply(outs_from_half_inning).rename("outs")
ip_by_game = (outs_by_game / 3.0).round(2).rename("IP_num")

# 3) 主客場與對手：若投手在「上半局」投球，則為主隊；多數票決定
def is_home_for_game(df):
    # inning_topbot: 'Top' or 'Bot'；主隊在 Top 投球
    # 以該場多數值決定（避免少數異常）
    top_ratio = (df["inning_topbot"].str.lower() == "top").mean()
    return 1 if top_ratio >= 0.5 else 0

home_flag = pitches.groupby("game_date").apply(is_home_for_game).rename("is_home")

# 取該場主客隊縮寫
teams = pitches.groupby("game_date")[["home_team", "away_team"]].agg(lambda x: x.iloc[0])
teams = teams.rename(columns={"home_team":"home_abbr","away_team":"away_abbr"})

# 對手隊伍縮寫
opp_team = pd.Series(
    np.where(home_flag.values==1, teams["away_abbr"].values, teams["home_abbr"].values),
    index=teams.index, name="opp_team"
)

# 4) 試算每場「失分（近似 ER）」：以對手分數的遞增量加總（近似，非正式 ER）
# Statcast 常有 home_score/away_score 欄位（得分變動），我們以分數增加量估算投手在場時的失分
def runs_allowed_estimate(df, is_home):
    # 選擇對手分數欄位
    score_col = "away_score" if is_home==1 else "home_score"
    if score_col not in df.columns:
        return np.nan
    sc = df[score_col].fillna(method="ffill").fillna(0)
    inc = sc.diff().clip(lower=0).fillna(0)
    return float(inc.sum())

runs_allowed = pitches.groupby("game_date").apply(
    lambda g: runs_allowed_estimate(g, is_home_for_game(g))
).rename("R_est")

# ------- 合併逐場匯總 -------
games = pd.concat([game_pcount, ip_by_game, home_flag, teams, opp_team, runs_allowed], axis=1).reset_index()

# 5) 近期 3 場平均
games = games.sort_values("game_date")
games["avg_ip_last3"] = games["IP_num"].rolling(3, min_periods=1).mean().round(2)
games["avg_er_last3"] = games["R_est"].rolling(3, min_periods=1).mean().round(2)

# 6) 休息天數
games["rest_days"] = games["game_date"].diff().dt.days
# 對第一場給個合理預設（例如 5 天）
games["rest_days"] = games["rest_days"].fillna(5).astype(int)

# 7) 最近 7 天累積投球數
games["cum_pitch_count_7d"] = [
    games.loc[(games["game_date"] >= d - timedelta(days=7)) & (games["game_date"] < d), "pitches"].sum()
    for d in games["game_date"]
]

# ========= 對手火力（OPS / wOBA） =========
tb = team_batting(YEAR).copy()  # FanGraphs 球隊打擊
# 嘗試用縮寫對齊：部分版本 Team 就是縮寫，否則請自行建立 mapping
tb.rename(columns={"Team": "opp_team"}, inplace=True)
# 若 Team 不是縮寫，可另外準備一個字典把 'New York Yankees'→'NYY' 之類做轉換

# 合併對手火力
games = games.merge(tb[["opp_team","OPS","wOBA"]], on="opp_team", how="left")
games.rename(columns={"OPS":"opp_ops","wOBA":"opp_woba"}, inplace=True)

# ========= 投手賽季屬性（ERA/WHIP/手別） =========
ps = pitching_stats(YEAR, YEAR, qual=0)
# 名稱在 FanGraphs 可能為 "First Last"；保險起見以姓氏包含過濾再挑選
ps_sel = ps[ps["Name"].str.contains(PITCHER_LAST, case=False, na=False)]
if ps_sel.empty:
    # 找不到就用整表再次以全名嘗試
    ps_sel = ps[ps["Name"] == f"{PITCHER_FIRST} {PITCHER_LAST}"]
if ps_sel.empty:
    raise RuntimeError("找不到投手的季級資料，請手動檢查名字在 FanGraphs 欄位的寫法。")

season_era = float(ps_sel["ERA"].iloc[0])
season_whip = float(ps_sel["WHIP"].iloc[0])

# 投手手別：直接取 Statcast 的 p_throws 眾數（R/L）
hand = pitches["p_throws"].dropna().mode()
hand = str(hand.iloc[0]) if not hand.empty else np.nan

games["season_era"] = season_era
games["season_whip"] = season_whip
games["hand"] = hand

# ========= 環境（park_factor / 天氣 / 主客場） =========
# 主客場已在 is_home。park_factor/天氣需外部來源；這裡先給預設或留空欄位，避免中斷。
# 你可以之後用 FanGraphs 的 Park Factors 表合併，或用 Open-Meteo 依日期+城市查天氣。
games["park_factor"] = np.nan    # 建議用 ballpark 對照表後續合併
games["temp_c"] = np.nan         # 建議外部氣象 API
games["humidity"] = np.nan       # 建議外部氣象 API

# ========= 產生 QS 標籤（用 R_est 近似 ER；若你有真實 ER，替換這一欄） =========
games["QS"] = ((games["IP_num"] >= 6) & (games["R_est"] <= 3)).astype(int)

# ========= 輸出欄位 =========
out_cols = [
    "game_date", "IP_num", "avg_ip_last3", "avg_er_last3",
    "rest_days", "cum_pitch_count_7d",
    "opp_team", "opp_ops", "opp_woba",
    "is_home", "park_factor", "temp_c", "humidity",
    "season_era", "season_whip", "hand", "pitches", "R_est", "home_abbr","away_abbr"
]
games[out_cols].to_excel(OUTFILE, index=False)
print(f"✅ Done. 輸出：{OUTFILE}")


In [ ]:
import pandas as pd

DATE = "2025-03-18"

# 取一天並排序（確保 diff 正確）
day = pitches.copy()
day["game_date"] = pd.to_datetime(day["game_date"])
day = day[day["game_date"].dt.strftime("%Y-%m-%d") == DATE].copy()

sort_cols = [c for c in ["game_pk","inning","inning_topbot","at_bat_number","pitch_number"] if c in day.columns]
day = day.sort_values(sort_cols)

# 確保 outs_when_up 是數值
day["outs_when_up"] = pd.to_numeric(day["outs_when_up"], errors="coerce")

# ✅ 关键：用 groupby().diff()（會保留原索引，不會出現不相容索引）
day["outs_delta"] = (
    day.groupby(["game_pk","inning_topbot"])["outs_when_up"]
       .diff()                          # 同半局內的變化
       .clip(lower=0)                   # 只保留正向增加
       .fillna(0)
       .astype(int)
)

# 只印你要的兩欄
pd.set_option("display.max_rows", None)      # 顯示所有列
pd.set_option("display.max_columns", None)   # 顯示所有欄
pd.set_option("display.width", 0)            # 讓輸出自動換行
pd.set_option("display.max_colwidth", None)  # 欄位內容完整顯示

print(day[["game_date", "inning", "outs_when_up", "outs_delta"]])
print("\n=== 每場彙總 ===")
print(pd.concat([outs_by_game, ip_by_game], axis=1))

# （可選）當天每場總出局數與投球局數
outs_by_game = day.groupby("game_pk")["outs_delta"].sum().rename("outs")
ip_by_game = (outs_by_game / 3.0).round(2).rename("IP_num")
print(pd.concat([outs_by_game, ip_by_game], axis=1))


In [ ]:
import pandas as pd

DATE = "2025-03-18"

# 取一天並排序（確保時序正確）
day = pitches.copy()
day["game_date"] = pd.to_datetime(day["game_date"])
day = day[day["game_date"].dt.strftime("%Y-%m-%d") == DATE].copy()

# 標準化半局標記
if "inning_topbot" in day.columns:
    day["inning_topbot"] = day["inning_topbot"].astype(str).str.strip().str.lower()

sort_cols = [c for c in ["game_pk","inning","inning_topbot","at_bat_number","pitch_number"] if c in day.columns]
day = day.sort_values(sort_cols)

# 先用「同一場+同半局」的正向 diff 做基礎的 outs_delta
day["outs_when_up"] = pd.to_numeric(day["outs_when_up"], errors="coerce").fillna(0)
group_keys = [k for k in ["game_pk","inning_topbot"] if k in day.columns]
day["outs_delta"] = (
    day.groupby(group_keys)["outs_when_up"]
       .diff().clip(lower=0).fillna(0).astype(int)
)
print(day["outs_delta"])

# ---- 簡單修正：補上半局最後一球的「第3個出局」 ----
# 若出現「上一列=2、下一列=0」且同一場，且半局或局數改變，就把前一列的 outs_delta +1
idx = day.index.to_list()
for i in range(1, len(day)):
    prev, cur = idx[i-1], idx[i]

    same_game = ("game_pk" in day.columns) and (day.at[prev, "game_pk"] == day.at[cur, "game_pk"])
    if not same_game:
        continue

    prev_out = day.at[prev, "outs_when_up"]
    cur_out  = day.at[cur, "outs_when_up"]

    # 半局/局數是否換了（任何一種變化都算半局結束）
    half_changed = False
    if "inning_topbot" in day.columns and day.at[prev,"inning_topbot"] != day.at[cur,"inning_topbot"]:
        half_changed = True
    if "inning" in day.columns and day.at[prev,"inning"] != day.at[cur,"inning"]:
        half_changed = True

    if prev_out == 2 and cur_out == 0 and half_changed:
        day.at[prev, "outs_delta"] += 1  # 給上一球補上第3個出局

# 只印你要看的兩欄（可加 inning 方便檢查）
print(day[["game_date","inning","outs_when_up","outs_delta"]].to_string(index=True))

# 當天每場的總出局與 IP
outs_by_game = day.groupby("game_pk")["outs_delta"].sum().rename("outs")
ip_by_game = (outs_by_game / 3.0).round(2).rename("IP_num")
print("\n=== 當天逐場合計 ===")
print(pd.concat([outs_by_game, ip_by_game], axis=1).to_string())
